# GPU Matrix Multiplication Optimization Tutorial

This notebook guides you through optimizing a matrix multiplication kernel from naive to **50+ TFLOPS**, progressively applying key GPU optimization techniques.

Special thanks: https://github.com/seb-v/fp32_sgemm_amd

### Environment Setup
Verify your ROCm/HIP environment:

In [9]:
!hipcc --version

HIP version: 7.2.26015-fc0010cf6a
AMD clang version 22.0.0git (https://github.com/RadeonOpenCompute/llvm-project roc-7.2.0 26014 7b800a19466229b8479a78de19143dc33c3ab9b5)
Target: x86_64-unknown-linux-gnu
Thread model: posix
InstalledDir: /opt/rocm-7.2.0/lib/llvm/bin
Configuration file: /opt/rocm-7.2.0/lib/llvm/bin/clang++.cfg


In [10]:
!rocminfo | grep -E 'Name|gfx'

  Name:                    AMD Ryzen 9 3900X 12-Core Processor
  Marketing Name:          AMD Ryzen 9 3900X 12-Core Processor
  Vendor Name:             CPU                                
  Name:                    gfx1100                            
  Marketing Name:          Radeon RX 7900 XTX                 
  Vendor Name:             AMD                                
      Name:                    amdgcn-amd-amdhsa--gfx1100         
      Name:                    amdgcn-amd-amdhsa--gfx11-generic   


---
## Problem: Matrix Multiplication C = A x B

For 4096x4096 FP32 matrices:
- **FLOPs**: 2 x 4096^3 = 137 trillion operations
- **Memory bandwidth needed** (naive): ~90.2 GB/s

## RDNA3 Memory Hierarchy
```
Global Memory (GMEM)    960 GB/s, high latency
         | load/store
LDS (Shared Mem)  ~29 TB/s, workgroup-shared (64KB per CU)
         | spill/fill
Registers     Per-thread, very fast (256 max per kernel)
```

---
## Baseline Performance Results (RX 7900 XTX)
| Kernel | Time | GFLOPS | vs rocBLAS |
|--------|------|------|----------|
| 0 - rocBLAS | 4.5ms | 30,547 | 100% |
| 1 - Naive | 136ms | 1,010 | 3.3% |
| 2 - LDS tiling | 34ms | 4,018 | 13.1% |
| 3 - Register tile | 6ms | 22,777 | 74.6% |
| 4 - GMEM dbl buffer | 5.4ms | 25,560 | 83.7% |
| 5 - LDS optimization | 4.1ms | 33,527 | **109.8%** |
| 6 - Dual FMA (ISA) | 3.6ms | 37,791 | **123.7%** |
| 7 - Loop unroll | 3.3ms | 41,256 | **135.1%** |
| 8 - Batched GMEM | 2.8ms | 49,047 | **160.6%** |

---
## Kernel 1: Naive Implementation
**Approach**: One thread per C[i,j] element.

```cpp
__global__ void kernel1_naive(const float* A, const float* B, float* C, int N) {
    int row = blockIdx.y * blockDim.y + threadIdx.y;
    int col = blockIdx.x * blockDim.x + threadIdx.x;
    if (row < N && col < N) {
        float acc = 0.0f;
        for (int k = 0; k < N; ++k)
            acc += A[row * K + k] * B[k * N + col];
        C[row * N + col] = acc;
    }
}
```

In [ ]:
!rocprofv3 --stats  --kernel-trace  -d prof_result --output-format csv --  ../fp32_sgemm_amd/sgemm -k 1

W20260313 16:28:16.648148 138120615944576 simple_timer.cpp:55] [rocprofv3] tool initialization ::     0.000557 sec
W20260313 16:28:16.652638 138120615944576 simple_timer.cpp:55] [rocprofv3] '../fp32_sgemm_amd/sgemm -k 1' ::     0.000000 sec
Running: Kernel 1 : Naive
W20260313 16:28:16.719362 138120615944576 tool.cpp:2422] HSA version 8.20.0 initialized (instance=0)
W20260313 16:28:18.080713 138120615944576 tool.cpp:3104] [PPID=237287][PID=245410][TID=245410][rocprofv3_error_signal_handler] rocprofv3 caught signal 2...
W20260313 16:28:18.080788 138120615944576 tool.cpp:3127] [PPID=237287][PID=245410][TID=245410][rocprofv3_error_signal_handler] rocprofv3 will wait for 0 children to exit
W20260313 16:28:18.080796 138120615944576 tool.cpp:3142] [PPID=237287][PID=245410][TID=245410][rocprofv3_error_signal_handler] rocprofv3 finalizing after signal 2...
^C


In [15]:
!rocprofv3 --hip-trace --pmc  --kernel-trace --pc-sampling-beta-enabled --output-format pftrace  -d prof_result --  ../fp32_sgemm_amd/sgemm -k 1

W20260313 16:38:18.672125 134973960946048 simple_timer.cpp:55] [rocprofv3] tool initialization ::     0.000849 sec
W20260313 16:38:18.672182 134973960946048 tool.cpp:2422] HIP (compiler) version 7.2.0 initialized (instance=0)
W20260313 16:38:18.682632 134973960946048 simple_timer.cpp:55] [rocprofv3] '../fp32_sgemm_amd/sgemm -k 1' ::     0.000000 sec
Running: Kernel 1 : Naive
W20260313 16:38:18.721540 134973960946048 tool.cpp:2422] HIP (runtime) version 7.2.0 initialized (instance=0)
W20260313 16:38:18.749346 134973960946048 tool.cpp:2422] HSA version 8.20.0 initialized (instance=0)
194.236 ms -> 707.846 GFLOPS
--------------------
W20260313 16:38:22.102778 134973960946048 simple_timer.cpp:55] [rocprofv3] '../fp32_sgemm_amd/sgemm -k 1' ::     3.420145 sec
[847.779]       perfetto.cc:47304 Configured tracing session 1, #sources:1, duration:0 ms, #buffers:1, total buffer size:1048576 KB, total sessions:1, uid:0 session name: ""
E20260313 16:38:22.598939 134973960946048 output_stream.cpp:1

**Analysis**:
- Low arithmetic intensity: 2N FLOPs / (3 x N x 4 bytes) = 0.167 F/byte
- Non-coalesced memory access on B[]
- Same A[row] loaded redundantly by many threads

---
## Kernel 2: LDS Tiling
**Optimization**: Load tile into shared memory (LDS), reuse across wave.

```cpp
#define TILE_SIZE 32
__global__ void kernel2_lds(const float *A, const float *B, float *C, int N)
{
    __shared__ float As[TILE_SIZE][TILE_SIZE];
    __shared__ float Bs[TILE_SIZE][TILE_SIZE];

    int row = blockIdx.y * TILE_SIZE + threadIdx.y;
    int col = blockIdx.x * TILE_SIZE + threadIdx.x;

    float sum = 0.0f;

    for (int t = 0; t < N; t += TILE_SIZE)
    {
        Bs[threadIdx.y][threadIdx.x] = B[N * (threadIdx.y + t) + col];
        As[threadIdx.y][threadIdx.x] = A[N * row + t + threadIdx.x];

        __syncthreads();

        for (int k = 0; k < TILE_SIZE; k++)
        {
            sum += As[threadIdx.y][k] * Bs[k][threadIdx.x];
        }

        __syncthreads();
    }

    if (row < N && col < N)
    {
        C[row * N + col] = sum;
    }
}
```

**Benefits**:
- Reuses data: Each A[i,k] used 32 times (once per thread in row)
- Arithmetic intensity: ~5.3 F/byte = 32x improvement
- Coalesced loads: Neighboring threads load contiguous GMEM

---
## Kernel 3: Register Tiling
**Optimization**: Further tile in registers, compute 64 elements per thread (8x8).

Key changes from Kernel 2:
- Compute **64 elements per thread** (BLOCK_SIZE_M x BLOCK_SIZE_N = 8x8)
- Accumulate partial sums in registers: `float r[8][8]`
- Only write to GMEM once at the end

**Benefits**:
- Less LDS traffic (no intermediate reads/writes)
- Higher ILP: More independent ops for out-of-order execution
- Better occupancy: Wave stays busy while waiting on memory

---
## Kernel 4: GMEM Double Buffering
**Optimization**: Hide load latency with two LDS tiles in flight.

```cpp
// Pre-load next tile while computing current
As[(t/BK+1)%2][threadIdx.y][threadIdx.x] = A[...];  // Load tile T+1
__syncthreads();
sum += As[t/BK%2][...] * Bs[...] ;                   // Compute with tile t
```

**Benefits**:
- Latency hiding: GMEM takes ~400 cycles; computation overlaps
- VALU utilization higher

---
## Kernel 5: LDS Optimization
**Optimizations**:
1. **Padding**: `__shared__ float As[BK][BM+4]` - avoid bank conflicts
2. **CU mode**: `-mcumode` - use smaller workgroup size (64 threads)
3. **Larger wavefront tile**: 16x8 instead of 8x8

**Bank conflicts**:
LDS has 32 banks, each 4 bytes. Without padding threads access same bank.
Solution: `As[BK][BM + PADDING_DIV_4]` to stagger accesses.

**Result**: 33,527 GFLOPS - first kernel to beat rocBLAS!

---
## Kernel 6: ISA-level Dual FMA
**Optimization**: Directly write AMD GCN assembly for precise control.

```s
; Dual FMA executes 2 independent FMAs in one cycle
v_dual_fmac_f32 v10, v11, s[6:7], v12, v13 :: 
             v_dual_fmac_f32 v15, v16, s[6:7], v17, v18
```

**Dual-issue constraints**:
- Different VGPR banks (bank = reg_id % 4)
- Source registers from different banks execute in parallel

**Result**: 37,791 GFLOPS (+13% vs Kernel 5)

---
## How to Run the Full Test Suite

In [ ]:
!cd ../fp32_sgemm_amd && ./build.sh

In [ ]:
! cd ../fp32_sgemm_amd && ./sgemm

---
## Profiling with rocprof
See what limits performance:

```bash
rocprofv3 --stats-db ./perf_stats.db ./sgemm
rocprof-parser stats.db | head -100
```

**Key metrics to watch**:
- GMEM throughput: Should approach 960 GB/s
- LDS utilization: Higher is better (80%+ ideal)
- VALU FMA32 Utilization: Target 75%+
- Wavefront stall reasons: Identify bottlenecks

---
## Summary: Optimization Journey

| Step | Technique | Key Insight |
|-----|---------|-------------|
| 1->2 | **LDS tiling** | Shared memory = fast cache for workgroup |
| 2->3 | Register tiling | Process multiple outputs per thread |
| 3->4 | **GMEM double buffer** | Overlap load and compute |
| 4->5 | **LDS bank fix + CU mode** | Avoid LDS stalls, better occupancy |
| 5->6 | **ISA dual FMA** | Precise register allocation |
| 6->7 | Loop unrolling | Schedule ops optimally |
| 7->8 | Batched loads | Reduce instruction overhead |

## References
- [Original Blog](https://seb-v.github.io/optimization/update/2025/01/20/Fast-GPU-Matrix-multiplication.html)
- [ROCm Documentation](https://rocm.docs.amd.com/)
- [RDNA3 ISA Guide](https://www.amd.com/en/developer/resources/rdna-architectures)